# VDS Strategies


## Setup

Log in so earthaccess can build an authenticated registry for indirect (HTTPS) access.

In [1]:
import earthaccess

earthaccess.login()

Auth(authenticated=True, user='earthaccess', strategy='netrc', system=urs.earthdata.nasa.gov)

## Case 1 — non-concatenatable granules


By default `virtualize()` uses `combine="nested"` and needs a `concat_dim` to stack granules. Some collections' granules do not share a dimension that can be concatenated — instead they must be aligned on their coordinate values, or given a brand-new index.

### 1a. Align by coordinates (`combine="by_coords"`)

`combine="by_coords"` lines the granules up on their shared coordinates (here the per-file `time` stamp) with an outer join.

In [2]:
sst_granules = earthaccess.search_data(
    concept_id="C1996881146-POCLOUD",
    temporal=("2025-01-01", "2025-01-10"),
)

vds = earthaccess.virtualize(sst_granules, concat_dim="time", access="indirect")
vds

/home/betolink/.micromamba/envs/refactoring-earthaccess/lib/python3.13/site-packages/virtualizarr/xarray.py:478: FutureWarning: In a future version, xarray will not decode the variable 'dt_1km_data' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  with xr.open_zarr(
/home/betolink/.micromamba/envs/refactoring-earthaccess/lib/python3.13/site-packages/virtualizarr/xarray.py:478: FutureWarning: In a future version, xarray will not decode the variable 'dt_1km_data' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instea

<xarray.Dataset> Size: 64GB
Dimensions:           (time: 11, lat: 17999, lon: 36000)
Coordinates:
  * time              (time) datetime64[ns] 88B 2025-01-01T09:00:00 ... 2025-...
  * lat               (lat) float32 72kB -89.99 -89.98 -89.97 ... 89.98 89.99
  * lon               (lon) float32 144kB -180.0 -180.0 -180.0 ... 180.0 180.0
Data variables:
    mask              (time, lat, lon) int8 7GB ManifestArray<shape=(11, 1799...
    sea_ice_fraction  (time, lat, lon) int8 7GB ManifestArray<shape=(11, 1799...
    dt_1km_data       (time, lat, lon) int8 7GB ManifestArray<shape=(11, 1799...
    analysed_sst      (time, lat, lon) int16 14GB ManifestArray<shape=(11, 17...
    analysis_error    (time, lat, lon) int16 14GB ManifestArray<shape=(11, 17...
    sst_anomaly       (time, lat, lon) int16 14GB ManifestArray<shape=(11, 17...
Attributes: (12/39)
    Conventions:                CF-1.7
    title:                      Daily MUR SST, Final product
    summary:                    A merged, multi-sensor L4 Foundation SST anal...
    references:                 http://podaac.jpl.nasa.gov/Multi-scale_Ultra-...
    institution:                Jet Propulsion Laboratory
    history:                    created at nominal 4-day latency; replaced nr...
    ...                         ...
    project:                    NASA Making Earth Science Data Records for Us...
    publisher_name:             GHRSST Project Office
    publisher_url:              http://www.ghrsst.org
    publisher_email:            ghrsst-po@nceo.ac.uk
    processing_level:           L4
    cdm_data_type:              grid

### 1b. Build a synthetic index with `preprocess`

If the granules have no dimension to stack, we can create one in `preprocess`. `preprocess` receives each single-granule dataset (so it can read global attributes, but not the file path). Here we promote a timestamp stored as a global attribute to a 1-D `time` index, then stack the granules along it with `combine="nested"`. We also mark `time` as a `loadable_variables` entry so it is materialized as a real array and can be used for label-based slicing.

In [6]:
import pandas as pd


def add_time_index(ds):
    # The timestamp lives in a global attribute (name varies by
    # collection: time_coverage_start, RangeBeginningDateTime, ...).
    date = pd.Timestamp(ds.attrs["time_coverage_start"])
    return ds.expand_dims("time").assign_coords(time=("time", [date]))

In [ ]:
vds = earthaccess.virtualize(
    sst_granules,
    combine="nested",
    concat_dim="time",
    preprocess=add_time_index,
    loadable_variables=["time"],
)
vds

Because `time` was loaded eagerly, it can be used to select by label without a kerchunk round-trip:

In [ ]:
vds.sel(time="2024-01-03")

## Case 2 — virtualize a single file


A single granule is opened directly with `open_virtual_dataset` (no combine machinery). `loadable_variables` materializes the coordinates we care about while the data variables stay virtual.

In [3]:
single = earthaccess.virtualize(
    [sst_granules[0]],
    access="indirect",
    loadable_variables=["time"],
)
single

/home/betolink/.micromamba/envs/refactoring-earthaccess/lib/python3.13/site-packages/virtualizarr/xarray.py:478: FutureWarning: In a future version, xarray will not decode the variable 'dt_1km_data' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  with xr.open_zarr(


<xarray.Dataset> Size: 6GB
Dimensions:           (time: 1, lat: 17999, lon: 36000)
Coordinates:
  * time              (time) datetime64[ns] 8B 2025-01-01T09:00:00
    lat               (lat) float32 72kB ManifestArray<shape=(17999,), dtype=...
    lon               (lon) float32 144kB ManifestArray<shape=(36000,), dtype...
Data variables:
    mask              (time, lat, lon) int8 648MB ManifestArray<shape=(1, 179...
    sea_ice_fraction  (time, lat, lon) int8 648MB ManifestArray<shape=(1, 179...
    dt_1km_data       (time, lat, lon) int8 648MB ManifestArray<shape=(1, 179...
    analysed_sst      (time, lat, lon) int16 1GB ManifestArray<shape=(1, 1799...
    analysis_error    (time, lat, lon) int16 1GB ManifestArray<shape=(1, 1799...
    sst_anomaly       (time, lat, lon) int16 1GB ManifestArray<shape=(1, 1799...
Attributes: (12/47)
    Conventions:                CF-1.7
    title:                      Daily MUR SST, Final product
    summary:                    A merged, multi-sensor L4 Foundation SST anal...
    references:                 http://podaac.jpl.nasa.gov/Multi-scale_Ultra-...
    institution:                Jet Propulsion Laboratory
    history:                    created at nominal 4-day latency; replaced nr...
    ...                         ...
    project:                    NASA Making Earth Science Data Records for Us...
    publisher_name:             GHRSST Project Office
    publisher_url:              http://www.ghrsst.org
    publisher_email:            ghrsst-po@nceo.ac.uk
    processing_level:           L4
    cdm_data_type:              grid

## Case 3 — return a DataTree (`tree=True`)


Some granules hold multiple HDF5/NetCDF4 groups (TEMPO NO2 has `product` and `geolocation` groups). `tree=True` returns a DataTree with one node per group instead of requiring a manual `xr.merge`.

In [21]:
import xarray as xr

tempo = earthaccess.search_data(
    short_name="TEMPO_NO2_L3",
    version="V03",
    temporal=("2025-01-11 12:00", "2025-01-18 12:00"),
    count=1,
)

files = earthaccess.download(tempo)
ds = xr.open_datatree(files[0])

In [22]:
ds

<xarray.DataTree>
Group: /
│   Dimensions:    (latitude: 2950, longitude: 7750, time: 1)
│   Coordinates:
│     * latitude   (latitude) float32 12kB 14.01 14.03 14.05 ... 72.95 72.97 72.99
│     * longitude  (longitude) float32 31kB -168.0 -168.0 -167.9 ... -13.03 -13.01
│     * time       (time) datetime64[ns] 8B 2025-01-11T12:52:43.027294720
│   Data variables:
│       weight     (latitude, longitude) float32 91MB ...
│   Attributes: (12/40)
│       history:                          2025-01-11T18:42:20Z: L2_regrid -v /tem...
│       scan_num:                         2
│       time_coverage_start:              2025-01-11T12:52:25Z
│       time_coverage_end:                2025-01-11T13:32:14Z
│       time_coverage_start_since_epoch:  1420635163.0272946
│       time_coverage_end_since_epoch:    1420637552.6548684
│       ...                               ...
│       title:                            TEMPO Level 3 nitrogen dioxide product
│       collection_shortname:             TEMPO_NO2_L3
│       collection_version:               1
│       keywords:                         EARTH SCIENCE>ATMOSPHERE>AIR QUALITY>NI...
│       summary:                          Nitrogen dioxide Level 3 files provide ...
│       coremetadata:                     \nGROUP                  = INVENTORYMET...
├── Group: /product
│       Dimensions:                                  (time: 1, latitude: 2950,
│                                                     longitude: 7750)
│       Data variables:
│           vertical_column_troposphere              (time, latitude, longitude) float64 183MB ...
│           vertical_column_troposphere_uncertainty  (time, latitude, longitude) float64 183MB ...
│           vertical_column_stratosphere             (time, latitude, longitude) float64 183MB ...
│           main_data_quality_flag                   (time, latitude, longitude) float32 91MB ...
├── Group: /qa_statistics
│       Dimensions:                                              (time: 1,
│                                                                 latitude: 2950,
│                                                                 longitude: 7750)
│       Data variables:
│           num_vertical_column_troposphere_samples              (time, latitude, longitude) float64 183MB ...
│           min_vertical_column_troposphere_sample               (time, latitude, longitude) float64 183MB ...
│           max_vertical_column_troposphere_sample               (time, latitude, longitude) float64 183MB ...
│           num_vertical_column_troposphere_uncertainty_samples  (time, latitude, longitude) float64 183MB ...
│           min_vertical_column_troposphere_uncertainty_sample   (time, latitude, longitude) float64 183MB ...
│           max_vertical_column_troposphere_uncertainty_sample   (time, latitude, longitude) float64 183MB ...
│           num_vertical_column_stratosphere_samples             (time, latitude, longitude) float64 183MB ...
│           min_vertical_column_stratosphere_sample              (time, latitude, longitude) float64 183MB ...
│           max_vertical_column_stratosphere_sample              (time, latitude, longitude) float64 183MB ...
│           num_vertical_column_total_samples                    (time, latitude, longitude) float64 183MB ...
│           min_vertical_column_total_sample                     (time, latitude, longitude) float64 183MB ...
│           max_vertical_column_total_sample                     (time, latitude, longitude) float64 183MB ...
├── Group: /geolocation
│       Dimensions:                 (time: 1, latitude: 2950, longitude: 7750)
│       Data variables:
│           solar_zenith_angle      (time, latitude, longitude) float32 91MB ...
│           viewing_zenith_angle    (time, latitude, longitude) float32 91MB ...
│           relative_azimuth_angle  (time, latitude, longitude) float32 91MB ...
└── Group: /support_data
        Dimensions:                            (time: 1, latitude: 2950, longitude: 7750

Navigate the groups just like any xarray DataTree:

In [23]:
tree = earthaccess.virtualize(
    [tempo[0]], access="indirect", group="/", parser="hdf5", tree=True
)
tree

<xarray.DataTree>
Group: /
│   Dimensions:    (longitude: 7750, latitude: 2950, time: 1)
│   Coordinates:
│     * longitude  (longitude) float32 31kB -168.0 -168.0 -167.9 ... -13.03 -13.01
│     * latitude   (latitude) float32 12kB 14.01 14.03 14.05 ... 72.95 72.97 72.99
│     * time       (time) datetime64[ns] 8B 2025-01-11T12:52:43.027294720
│   Data variables:
│       weight     (latitude, longitude) float32 91MB ManifestArray<shape=(2950, ...
│   Attributes: (12/40)
│       history:                          2025-01-11T18:42:20Z: L2_regrid -v /tem...
│       scan_num:                         2
│       time_coverage_start:              2025-01-11T12:52:25Z
│       time_coverage_end:                2025-01-11T13:32:14Z
│       time_coverage_start_since_epoch:  1420635163.0272946
│       time_coverage_end_since_epoch:    1420637552.6548684
│       ...                               ...
│       title:                            TEMPO Level 3 nitrogen dioxide product
│       collection_shortname:             TEMPO_NO2_L3
│       collection_version:               1
│       keywords:                         EARTH SCIENCE>ATMOSPHERE>AIR QUALITY>NI...
│       summary:                          Nitrogen dioxide Level 3 files provide ...
│       coremetadata:                     \nGROUP                  = INVENTORYMET...
├── Group: /product
│       Dimensions:                                  (time: 1, latitude: 2950,
│                                                     longitude: 7750)
│       Data variables:
│           vertical_column_troposphere              (time, latitude, longitude) float64 183MB ManifestArray<shape=(1, 2950, 7750), dtype=float64, chunks=(1, 7...
│           vertical_column_troposphere_uncertainty  (time, latitude, longitude) float64 183MB ManifestArray<shape=(1, 2950, 7750), dtype=float64, chunks=(1, 7...
│           vertical_column_stratosphere             (time, latitude, longitude) float64 183MB ManifestArray<shape=(1, 2950, 7750), dtype=float64, chunks=(1, 7...
│           main_data_quality_flag                   (time, latitude, longitude) int16 46MB ManifestArray<shape=(1, 2950, 7750), dtype=int16, chunks=(1, 984,...
├── Group: /qa_statistics
│       Dimensions:                                              (time: 1,
│                                                                 latitude: 2950,
│                                                                 longitude: 7750)
│       Data variables:
│           num_vertical_column_troposphere_samples              (time, latitude, longitude) int32 91MB ManifestArray<shape=(1, 2950, 7750), dtype=int32, chu...
│           min_vertical_column_troposphere_sample               (time, latitude, longitude) float64 183MB ManifestArray<shape=(1, 2950, 7750), dtype=float64, ...
│           max_vertical_column_troposphere_sample               (time, latitude, longitude) float64 183MB ManifestArray<shape=(1, 2950, 7750), dtype=float64, ...
│           num_vertical_column_troposphere_uncertainty_samples  (time, latitude, longitude) int32 91MB ManifestArray<shape=(1, 2950, 7750), dtype=int32, chu...
│           min_vertical_column_troposphere_uncertainty_sample   (time, latitude, longitude) float64 183MB ManifestArray<shape=(1, 2950, 7750), dtype=float64, ...
│           max_vertical_column_troposphere_uncertainty_sample   (time, latitude, longitude) float64 183MB ManifestArray<shape=(1, 2950, 7750), dtype=float64, ...
│           num_vertical_column_stratosphere_samples             (time, latitude, longitude) int32 91MB ManifestArray<shape=(1, 2950, 7750), dtype=int32, chu...
│           min_vertical_column_stratosphere_sample              (time, latitude, longitude) float64 183MB ManifestArray<shape=(1, 2950, 7750), dtype=float64, ...
│           max_vertical_column_stratosphere_sample              (time, latitude, longitude) float64 183MB ManifestArray<shape=(1, 2950, 7750), dtype=float64, ...
│           num_vertical_column_total_samples                    (time, 

## Summary

| Scenario | How |
| --- | --- |
| Granules can't be concatenated | `combine="by_coords"` + `join` |
| Need an index not in the file | `preprocess` + `combine="nested"` + `loadable_variables` |
| Single file | `virtualize([granule], loadable_variables=[...])` |
| Multi-group HDF5 | `virtualize([granule], tree=True)` |